# DATA Loading

In [ ]:
# Target files for this walkthrough
MODEL_DIR = Path("/home/amkapoor/LaBRAM_imple/Sub_EEG/logs/labram_dora_run_2")
TARGET_EEG_FILE = "/media/hdd1/amkapoor/N400/N400Stimset_manuscriptdata/N400Stimset_manuscriptdata/N400_epoched/sub-01/sub-01-_-NPC_aisle.wav.npy"
GROUND_TRUTH_AUDIO = "/media/hdd1/amkapoor/N400/N400Stimset_manuscriptdata/N400Stimset_manuscriptdata/stimuli/NPC_aisle.wav"

print(f"Using model from: {MODEL_DIR}")
print(f"Target EEG: {TARGET_EEG_FILE}")
print(f"Ground Truth Audio: {GROUND_TRUTH_AUDIO}")

## Preprocess raw EEG

In [ ]:
from mne.preprocessing import ICA
import warnings
warnings.filterwarnings("ignore")
def load_and_preprocess(raw_path: Path):
    raw = mne.io.read_raw_bdf(raw_path, preload=True, verbose='ERROR')
    # Keep only the first 128 EEG channels (BioSemi cap)
    raw = raw.pick_channels(raw.ch_names[:128])
    raw.set_eeg_reference('average', projection=False)
    raw.notch_filter(freqs=[60], verbose='ERROR')
    raw.filter(0.5, 50.0, verbose='ERROR')

    # ICA for blink removal
    ica = ICA(n_components=20, random_state=97, max_iter='auto')
    ica.fit(raw.copy().filter(1., 30., verbose='ERROR'))
    eog_chs = [ch for ch in raw.ch_names if 'EOG' in ch.upper()]
    if eog_chs:
        blink_idx, _ = ica.find_bads_eog(raw, ch_name=eog_chs[0])
    else:
        blink_idx = []  # no EOG channel available; skip blink removal
    ica.exclude = blink_idx
    raw_clean = ica.apply(raw.copy())

    raw_clean.resample(TARGET_EEG_SR)
    return raw_clean


def segment_and_save(raw, events_tsv: Path, out_dir: Path):
    df = pd.read_csv(events_tsv, sep='	')
    df = df[df['stim_onset(s)'].notna()].reset_index(drop=True)

    saved = []
    for i, row in df.iterrows():
        start = float(row['stim_onset(s)'])
        dur = float(row['stim_dur(s)'])
        end = start + dur
        if end > raw.times[-1]:
            continue
        epoch = raw.copy().crop(tmin=start, tmax=end)
        data = epoch.get_data()  # shape: (channels, time)
        fname = f"epoch_{i:03d}_{row['trial_type']}.npy"
        np.save(out_dir / fname, data.astype('float32'))
        saved.append({
            'file': fname,
            'trial_type': row['trial_type'],
            'stim_file': row.get('stim_file', ''),
            'frames': data.shape[-1],
            'seconds': data.shape[-1] / raw.info['sfreq']
        })
    return saved


## Run preprocessing + epoch export

In [11]:
RUN_PREPROCESS = True 
import warnings
warnings.filterwarnings("ignore")
if RUN_PREPROCESS:
    raw = load_and_preprocess(RAW_BDF_PATH)
    sfreq = raw.info['sfreq']
    saved_meta = segment_and_save(raw, EVENTS_TSV_PATH, EPOCH_NPY_DIR)
    print(f"Saved {len(saved_meta)} epochs to {EPOCH_NPY_DIR} @ {sfreq} Hz")
else:
    sfreq = TARGET_EEG_SR
    print("Skipping preprocessing for now; flip RUN_PREPROCESS to True to generate .npy epochs.")

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Fitting ICA to data using 128 channels (please be patient, this may take a while)
Selecting by number: 20 components
Fitting ICA took 74.4s.
Applying ICA to Raw instance
    Transforming to ICA space (20 components)
    Zeroing out 0 ICA components
    Projecting back using 128 PCA components
Saved 915 epochs to notebooks/epochs_npy @ 256.0 Hz


In [18]:
from glob import glob

sample_paths = sorted(EPOCH_NPY_DIR.glob('*.npy'))
assert sample_paths, f"No npy files found in {EPOCH_NPY_DIR}. Run the preprocessing cell first."

sample_path = sample_paths[10]
sample = np.load(sample_path)
print(f"Loaded {sample_path.name}: shape={sample.shape} (channels x frames)")
print(f"EEG sampling rate: {sfreq} Hz | frames: {sample.shape[-1]} (~{sample.shape[-1]/sfreq:.2f} s)")

Loaded epoch_010_Ex.npy: shape=(128, 521) (channels x frames)
EEG sampling rate: 256.0 Hz | frames: 521 (~2.04 s)


## Load LaBraM-based EEG→Speech model + checkpoint

In [19]:
# Helper to pick the newest common checkpoint step shared by G_* (generator) and E_* (EEG encoder)
def latest_common_step(model_dir: Path):
    steps = {
        prefix: {int(p.stem.split('_')[1]) for p in model_dir.glob(f"{prefix}_*.pth")}
        for prefix in ['G', 'E']
    }
    common = sorted(steps['G'].intersection(steps['E']))
    if not common:
        raise RuntimeError('No overlapping G_* and E_* checkpoints found')
    return common[-1]

hps = get_hparams_from_file(str(MODEL_DIR / 'configs.json'))
ckpt_step = latest_common_step(MODEL_DIR)
print(f"Using checkpoint step {ckpt_step} from {MODEL_DIR}")

eeg_cfg = hps.model.eeg_module

eeg_module = EEGModule(
    n_layers_cnn=eeg_cfg.n_layers_cnn,
    use_s4=eeg_cfg.use_s4,
    n_layers_s4=eeg_cfg.n_layers_s4,
    embedding_size=hps.model.inter_channels,
    is_mask=False,
    in_channels=eeg_cfg.in_channels,
    num_subjects=getattr(hps.model, 'num_subjects', 25),
    use_labram=getattr(eeg_cfg, 'use_labram', False),
    labram_checkpoint=getattr(eeg_cfg, 'labram_checkpoint', None),
    use_dora=getattr(eeg_cfg, 'use_dora', False),
    lora_rank=getattr(eeg_cfg, 'lora_rank', 8),
    lora_alpha=getattr(eeg_cfg, 'lora_alpha', 16),
    device=device
).to(device).eval()

speech_decoder = SpeechDecoder(
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model.__dict__
).to(device).eval()

load_checkpoint(str(MODEL_DIR / f'G_{ckpt_step}.pth'), speech_decoder)
load_checkpoint(str(MODEL_DIR / f'E_{ckpt_step}.pth'), eeg_module)

Using checkpoint step 244000 from logs/labram_dora_run_2
Initializing EEGModule with LaBraM backbone...
Loading LaBraM weights from /home/amkapoor/LaBRAM_imple/Sub_EEG/labram-base.pth
LaBraM loaded with msg: <All keys matched successfully>
Applying DoRA to LaBraM (rank=8, alpha=16)...
trainable params: 288,000 || all params: 6,107,936 || trainable%: 4.7152
INFO:root:Loaded checkpoint 'logs/labram_dora_run_2/G_244000.pth' (iteration 1085)
INFO:root:Loaded checkpoint 'logs/labram_dora_run_2/E_244000.pth' (iteration 1085)


(EEGModule(
   (labram_backbone): PeftModel(
     (base_model): LoraModel(
       (model): NeuralTransformer(
         (patch_embed): TemporalConv(
           (conv1): Conv2d(1, 8, kernel_size=(1, 15), stride=(1, 8), padding=(0, 7))
           (gelu1): GELU(approximate='none')
           (norm1): GroupNorm(4, 8, eps=1e-05, affine=True)
           (conv2): Conv2d(8, 8, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
           (gelu2): GELU(approximate='none')
           (norm2): GroupNorm(4, 8, eps=1e-05, affine=True)
           (conv3): Conv2d(8, 8, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
           (norm3): GroupNorm(4, 8, eps=1e-05, affine=True)
           (gelu3): GELU(approximate='none')
         )
         (pos_drop): Dropout(p=0.0, inplace=False)
         (blocks): ModuleList(
           (0-11): 12 x Block(
             (norm1): LayerNorm((200,), eps=1e-06, elementwise_affine=True)
             (attn): Attention(
               (qkv): lora.Linear(
                 

## Shapes from EEG → latent → audio waveform

In [20]:
@torch.no_grad()
def run_trace(eeg_np: np.ndarray):
    print("--- LaBRAM Architecture Shape Analysis ---")
    eeg_tensor = torch.tensor(eeg_np, dtype=torch.float32, device=device)
    if eeg_tensor.ndim == 2:
        eeg_tensor = eeg_tensor.unsqueeze(0)
    
    print(f"1. Input EEG shape: {eeg_tensor.shape} (Batch, Channels, Time)")
    
    # EEG Encoder tracing
    # x_raw is the CNN output, mid_output is the S4/Transformer output
    x_raw, mask, mid_output, eeg_recon, sid_logits = eeg_module(eeg_tensor)
    
    if hasattr(eeg_module, 'encoder'):
         print(f"2. CNN Encoder Output (x_raw): {x_raw.shape}")
    
    print(f"3. Latent Bottleneck (mid_output): {mid_output.shape} (Batch, Hidden, Time)")
    
    if eeg_recon is not None:
        print(f"4. Reconstruction Decoder Output: {eeg_recon.shape}")
    
    mid_lengths = torch.tensor([mid_output.size(2)], device=device)

    # Speech Decoder tracing
    # Connector output (Transformer encoder stats) before sampling
    conn_x, m_p, logs_p, x_mask = speech_decoder.enc_proj(mid_output, mid_lengths)
    print(f"5. Speech Connector Output (conn_x): {conn_x.shape}")
    print(f"6. Prior Mean (m_p): {m_p.shape}, Log-Var (logs_p): {logs_p.shape}")

    max_len = max(1, int(mid_lengths.max().item()))
    y_hat, _, y_mask, (z, z_p, m_p_inf, logs_p_inf), _ = speech_decoder.infer(
        mid_output, mid_lengths, max_len=max_len, noise_scale=0.667
    )

    print(f"7. Flow Latents z: {z.shape}")
    print(f"8. Synthesized Waveform (y_hat): {y_hat.shape} (Batch, 1, Samples)")
    
    audio_samples = y_hat.shape[-1]
    duration_sec = audio_samples / AUDIO_SR
    print(f"   - Result Duration: {duration_sec:.2f} s @ {AUDIO_SR} Hz")
    
    return y_hat.cpu(), duration_sec

synth_wav, synth_duration = run_trace(sample)


EEG input: torch.Size([1, 128, 521]) | sfreq=256 Hz | frames=521
Latent mid_output: torch.Size([1, 192, 175]) (frames=175)
Connector output x: torch.Size([1, 192, 175]) | m/logs: torch.Size([1, 192, 175])
EEG recon decoder output: torch.Size([1, 128, 518])
Flow latents z: torch.Size([1, 192, 175]), z_p: torch.Size([1, 192, 175]), m_p: torch.Size([1, 192, 175]), logs_p: torch.Size([1, 192, 175])
Audio waveform: torch.Size([1, 1, 44800]) => 44800 samples @ 22050 Hz ( = 2.03 s)
